# 🧪 Automação QA Roteiro PDV — v3

Valida roteiros de teste do PDV Scanntech cruzando três fontes:
- 📋 **Planilha** do roteiro (obrigatório em todas as etapas)
- 📄 **JSON de venda** — movimentos retornados pela API (Etapa 2+)
- 🎟️ **Cupons / promoções** — JSON da consulta de promoções (Etapa 3+)

### Etapas disponíveis
| Etapa | O que valida |
|-------|-------------|
| **1 — Planilha (MVP)** | Leitura, parse de itens, pagamento, subtotal/desconto/total |
| **2 — Planilha + JSON** | Cruza EANs, quantidades, total e pagamentos com o movimento |
| **3 — Planilha + JSON + Cupons** | Adiciona validação de tipo de promo, limite e BIN |


In [ ]:
# ── Célula 1: Instalação ─────────────────────────────────────────────────────
!pip install -q pandas openpyxl

In [ ]:
# ── Célula 2: Clonar / atualizar repositório ─────────────────────────────────
import os, sys

if os.path.exists('automacao_scann'):
    %cd automacao_scann
    !git pull origin main
else:
    !git clone https://github.com/Miked0/automacao_scann.git
    %cd automacao_scann

sys.path.insert(0, 'src')
print('✅ Repositório pronto')

---
## ⚙️ Configuração — Escolha a etapa de validação

Edite a variável `ETAPA` abaixo antes de continuar:
- `1` → só planilha
- `2` → planilha + JSON de venda
- `3` → planilha + JSON de venda + cupons/promoções

In [ ]:
# ── Célula 3: Configuração da etapa ──────────────────────────────────────────

# ✏️  EDITE AQUI: 1, 2 ou 3
ETAPA = 1

# ─────────────────────────────────────────────────────────────────────────────
assert ETAPA in (1, 2, 3), 'ETAPA deve ser 1, 2 ou 3'

ETAPA_LABELS = {
    1: 'Etapa 1 — Planilha (MVP)',
    2: 'Etapa 2 — Planilha + JSON de Venda',
    3: 'Etapa 3 — Planilha + JSON de Venda + Cupons/Promoções',
}
print(f'🎯 Modo selecionado: {ETAPA_LABELS[ETAPA]}')

---
## 📂 Upload dos arquivos

A próxima célula vai pedir os uploads conforme a etapa escolhida.

In [ ]:
# ── Célula 4: Upload inteligente por etapa ───────────────────────────────────
from google.colab import files
import shutil, os, json

os.makedirs('input', exist_ok=True)
os.makedirs('output', exist_ok=True)

INPUT_PLANILHA = None
INPUT_JSON = None
INPUT_CUPONS = None

# ── 1. Planilha (obrigatório em todas as etapas) ──────────────────────────────
print('=' * 60)
print('📋 PASSO 1/3 — Upload da planilha do roteiro (.xlsx)')
print('=' * 60)
uploaded = files.upload()
for fname in uploaded:
    dest = f'input/{fname}'
    shutil.copy(fname, dest)
    INPUT_PLANILHA = dest
    print(f'  ✅ Planilha: {dest}')

# ── 2. JSON de movimentos (Etapa 2 e 3) ──────────────────────────────────────
if ETAPA >= 2:
    print()
    print('=' * 60)
    print('📄 PASSO 2/3 — Upload do JSON de venda (movimientos)')
    print('    Arquivo gerado pelo relatório do PDV ou exportado da API')
    print('=' * 60)
    uploaded_json = files.upload()
    for fname in uploaded_json:
        dest = f'input/{fname}'
        shutil.copy(fname, dest)
        INPUT_JSON = dest
        print(f'  ✅ JSON de venda: {dest}')
    # Valida estrutura mínima do JSON
    with open(INPUT_JSON) as f:
        _data = json.load(f)
    if isinstance(_data, list):
        print(f'  📊 {len(_data)} movimento(s) encontrado(s) no JSON')
    elif isinstance(_data, dict):
        print(f'  📊 JSON carregado (objeto único ou wrapper)')
else:
    print()
    print('⏭️  PASSO 2/3 — JSON de venda não necessário na Etapa 1')

# ── 3. Cupons / Promoções (Etapa 3) ──────────────────────────────────────────
if ETAPA >= 3:
    print()
    print('=' * 60)
    print('🎟️  PASSO 3/3 — Upload do JSON de cupons/promoções')
    print('    Resultado da consulta de promoções da API Scanntech')
    print('=' * 60)
    uploaded_cupons = files.upload()
    for fname in uploaded_cupons:
        dest = f'input/{fname}'
        shutil.copy(fname, dest)
        INPUT_CUPONS = dest
        print(f'  ✅ Cupons/Promoções: {dest}')
else:
    print()
    print('⏭️  PASSO 3/3 — Cupons/Promoções não necessários nesta etapa')

print()
print('─' * 60)
print(f'📋 Planilha : {INPUT_PLANILHA}')
print(f'📄 JSON Venda: {INPUT_JSON}')
print(f'🎟️  Cupons   : {INPUT_CUPONS}')
print('─' * 60)

---
## ▶️ Executar validação

In [ ]:
# ── Célula 5: Executar validação conforme etapa ───────────────────────────────
from main import run_mvp
import json, pandas as pd
from reader import load_roteiro_tests
from validators import validate_test_case
from exporters import export_results
from pathlib import Path

OUTPUT_FILE = 'output/validacao_resultado.xlsx'

print(f'🚀 Iniciando validação — {ETAPA_LABELS[ETAPA]}')
print()

# ── Etapa 1: só planilha ─────────────────────────────────────────────────────
if ETAPA == 1:
    run_mvp(INPUT_PLANILHA, OUTPUT_FILE)

# ── Etapa 2: planilha + JSON de venda ────────────────────────────────────────
elif ETAPA == 2:
    tests = load_roteiro_tests(Path(INPUT_PLANILHA))
    with open(INPUT_JSON) as f:
        movimentos_raw = json.load(f)
    # Normaliza: aceita lista ou {'movimientos': [...]} ou {'data': [...]}
    if isinstance(movimentos_raw, list):
        movimentos = movimentos_raw
    elif isinstance(movimentos_raw, dict):
        movimentos = (movimentos_raw.get('movimientos')
                      or movimentos_raw.get('data')
                      or movimentos_raw.get('items')
                      or [movimentos_raw])
    else:
        movimentos = []
    print(f'  📊 {len(movimentos)} movimento(s) carregado(s)')
    # Indexa por numero do movimento para cruzamento rápido
    idx_mov = {str(m.get('numero', m.get('id', i))): m for i, m in enumerate(movimentos)}
    results = []
    for t in tests:
        base = validate_test_case(t)
        # Tenta cruzar pelo número do teste
        num_teste = str(t.get('teste', '')).strip()
        mov = idx_mov.get(num_teste)
        if mov:
            # Total
            from decimal import Decimal
            tot_json = Decimal(str(mov.get('total', mov.get('importeTotal', 0))))
            tot_plan = Decimal(str(base.get('total_norm') or 0))
            diff_total = (tot_json - tot_plan).copy_abs()
            base['json_total'] = str(tot_json)
            base['json_diff_total'] = str(diff_total)
            base['json_status'] = 'OK' if diff_total <= Decimal('0.01') else 'DIVERGENCIA'
            # Desconto
            dsc_json = Decimal(str(mov.get('descuentoTotal', mov.get('desconto', 0))))
            dsc_plan = Decimal(str(base.get('desconto_norm') or 0))
            base['json_desconto'] = str(dsc_json)
            base['json_diff_desconto'] = str((dsc_json - dsc_plan).copy_abs())
            # EANs
            from parser_items import parse_itens
            itens_plan = parse_itens(t.get('itens_raw'))
            eans_plan = {i['codigo'] for i in itens_plan if i['tipo'] == 'ean'}
            detalles = mov.get('detalles', mov.get('items', []))
            eans_json = {str(d.get('codigoBarras', d.get('ean', ''))) for d in detalles}
            faltando = eans_plan - eans_json
            extra = eans_json - eans_plan
            base['json_eans_faltando'] = str(faltando) if faltando else None
            base['json_eans_extra'] = str(extra) if extra else None
            if faltando or extra or base['json_status'] == 'DIVERGENCIA':
                if base['status_final'] == 'OK':
                    base['status_final'] = 'DIVERGENCIA_JSON'
        else:
            base['json_status'] = 'SEM_MATCH'
            base['json_total'] = None
            base['json_desconto'] = None
            base['json_eans_faltando'] = None
            base['json_eans_extra'] = None
            base['json_diff_total'] = None
            base['json_diff_desconto'] = None
        results.append({**t, **base})
    export_results(results, Path(OUTPUT_FILE))

# ── Etapa 3: planilha + JSON + cupons ────────────────────────────────────────
elif ETAPA == 3:
    # Reutiliza lógica da etapa 2 + cruza cupons
    tests = load_roteiro_tests(Path(INPUT_PLANILHA))
    with open(INPUT_JSON) as f:
        movimentos_raw = json.load(f)
    if isinstance(movimentos_raw, list):
        movimentos = movimentos_raw
    elif isinstance(movimentos_raw, dict):
        movimentos = (movimentos_raw.get('movimientos')
                      or movimentos_raw.get('data')
                      or [movimentos_raw])
    with open(INPUT_CUPONS) as f:
        cupons_raw = json.load(f)
    if isinstance(cupons_raw, list):
        cupons = cupons_raw
    elif isinstance(cupons_raw, dict):
        cupons = (cupons_raw.get('promociones')
                  or cupons_raw.get('promotions')
                  or cupons_raw.get('data')
                  or [cupons_raw])
    print(f'  📊 {len(movimentos)} movimento(s) | 🎟️ {len(cupons)} cupom/promoção(ões)')
    idx_mov = {str(m.get('numero', m.get('id', i))): m for i, m in enumerate(movimentos)}
    idx_cup = {}
    for c in cupons:
        key = str(c.get('numero', c.get('id', c.get('movimiento', ''))))
        idx_cup[key] = c
    results = []
    for t in tests:
        from decimal import Decimal
        from parser_items import parse_itens
        base = validate_test_case(t)
        num_teste = str(t.get('teste', '')).strip()
        mov = idx_mov.get(num_teste)
        cup = idx_cup.get(num_teste)
        # Cruzamento com JSON de venda
        if mov:
            tot_json = Decimal(str(mov.get('total', mov.get('importeTotal', 0))))
            tot_plan = Decimal(str(base.get('total_norm') or 0))
            diff_total = (tot_json - tot_plan).copy_abs()
            base['json_total'] = str(tot_json)
            base['json_diff_total'] = str(diff_total)
            base['json_status'] = 'OK' if diff_total <= Decimal('0.01') else 'DIVERGENCIA'
            dsc_json = Decimal(str(mov.get('descuentoTotal', mov.get('desconto', 0))))
            base['json_desconto'] = str(dsc_json)
            base['json_diff_desconto'] = str((dsc_json - Decimal(str(base.get('desconto_norm') or 0))).copy_abs())
            itens_plan = parse_itens(t.get('itens_raw'))
            eans_plan = {i['codigo'] for i in itens_plan if i['tipo'] == 'ean'}
            detalles = mov.get('detalles', mov.get('items', []))
            eans_json = {str(d.get('codigoBarras', d.get('ean', ''))) for d in detalles}
            faltando = eans_plan - eans_json
            extra = eans_json - eans_plan
            base['json_eans_faltando'] = str(faltando) if faltando else None
            base['json_eans_extra'] = str(extra) if extra else None
            if faltando or extra or base['json_status'] == 'DIVERGENCIA':
                if base['status_final'] == 'OK':
                    base['status_final'] = 'DIVERGENCIA_JSON'
        else:
            base['json_status'] = 'SEM_MATCH'
            base['json_total'] = None
            base['json_desconto'] = None
            base['json_eans_faltando'] = None
            base['json_eans_extra'] = None
            base['json_diff_total'] = None
            base['json_diff_desconto'] = None
        # Cruzamento com cupons/promoções
        if cup:
            tipo_promo_plan = str(t.get('tipo_promo', '') or '').strip().upper()
            tipo_promo_json = str(cup.get('tipoPromocion', cup.get('tipo', cup.get('type', '')))).upper()
            base['cupon_tipo_promo_json'] = tipo_promo_json
            base['cupon_tipo_match'] = 'OK' if tipo_promo_plan in tipo_promo_json or tipo_promo_json in tipo_promo_plan else 'DIVERGENCIA'
            # Limite por ticket
            limite_json = cup.get('limitePromocionesPorTicket', cup.get('limite'))
            base['cupon_limite_json'] = str(limite_json) if limite_json is not None else None
            # Forma de pagamento exigida
            formas_pago = cup.get('formasPago', cup.get('paymentMethods', []))
            base['cupon_formas_pago'] = str(formas_pago) if formas_pago else None
            # BIN
            bines = cup.get('bines', cup.get('bins', []))
            base['cupon_bines'] = str(bines) if bines else None
            if base['cupon_tipo_match'] == 'DIVERGENCIA':
                if base['status_final'] == 'OK':
                    base['status_final'] = 'DIVERGENCIA_CUPON'
        else:
            base['cupon_tipo_promo_json'] = None
            base['cupon_tipo_match'] = 'SEM_MATCH'
            base['cupon_limite_json'] = None
            base['cupon_formas_pago'] = None
            base['cupon_bines'] = None
        results.append({**t, **base})
    export_results(results, Path(OUTPUT_FILE))

print()
print('✅ Validação concluída!')

---
## 📊 Resumo do resultado

In [ ]:
# ── Célula 6: Resumo visual ───────────────────────────────────────────────────
import pandas as pd

df = pd.read_excel(OUTPUT_FILE, sheet_name='Resultado')

print(f'🎯 {ETAPA_LABELS[ETAPA]}')
print('=' * 60)
print(f'Total de casos: {len(df)}')
print()
print('── Status final ──')
print(df['status_final'].value_counts().to_string())

if ETAPA >= 2 and 'json_status' in df.columns:
    print()
    print('── Status cruzamento JSON ──')
    print(df['json_status'].value_counts().to_string())

if ETAPA >= 3 and 'cupon_tipo_match' in df.columns:
    print()
    print('── Status cruzamento Cupons ──')
    print(df['cupon_tipo_match'].value_counts().to_string())

print()
cols_show = ['bloco', 'teste', 'tipo_promo', 'status_final', 'motivo_status', 'alertas']
if ETAPA >= 2:
    cols_show += ['json_status', 'json_diff_total', 'json_eans_faltando']
if ETAPA >= 3:
    cols_show += ['cupon_tipo_match', 'cupon_limite_json', 'cupon_bines']
cols_show = [c for c in cols_show if c in df.columns]
df[cols_show]

In [ ]:
# ── Célula 7: Download do resultado ──────────────────────────────────────────
from google.colab import files
files.download(OUTPUT_FILE)
print('📥 Download iniciado!')